In [1]:
import pandas as pd
pd.set_option('display.float_format', lambda x: '{:,.2f}'.format(x).replace(',', ' '))

In [2]:
df  = pd.read_excel('...Тестовое_задание_Data_аналитик_ЦО_2025_2.xlsx',
                    sheet_name = 'Данные для задачи 3').sort_values(by=['Бренд AX 1', 'Неделя'])

In [3]:
pivot_df = df.pivot_table(
    index='Бренд AX 1', 
    columns='Неделя', 
    values=['Выручка', 'Маржинальность'],
    aggfunc='sum'
).reset_index().copy()

In [4]:
pivot_df.columns = [f'{col[0]}_{col[1]}' for col in pivot_df.columns]

In [19]:
total_revenue_14 = pivot_df['Выручка_14'].sum().round()
total_revenue_15 = pivot_df['Выручка_15'].sum().round()

margin_rub_14 = pivot_df['Выручка_14'] * pivot_df['Маржинальность_14'] 
margin_rub_15 = pivot_df['Выручка_15'] * pivot_df['Маржинальность_15']

total_margin_rub_14 = margin_rub_14.sum().round()
total_margin_rub_15 = margin_rub_15.sum().round()

total_margin_pct_14 = (total_margin_rub_14 / total_revenue_14) * 100
total_margin_pct_15 = (total_margin_rub_15 / total_revenue_15) * 100

diff_revenue = (total_revenue_15 - total_revenue_14) / total_revenue_14 * 100
diff_margin = (total_margin_rub_15 - total_margin_rub_14) / total_margin_rub_14 * 100

print(f"Неделя 14")
print(f"Общая выручка: {total_revenue_14:,}")
print(f"Общая маржа (в деньгах): {total_margin_rub_14:,}")
print(f"Итоговая маржинальность: {total_margin_pct_14:.2f}%")

print(f"\nНеделя 15")
print(f"Общая выручка: {total_revenue_15:,}")
print(f"Общая маржа (в деньгах): {total_margin_rub_15:,}")
print(f"Итоговая маржинальность: {total_margin_pct_15:.2f}%")

print(f"\nDiff")
print(f"Выручка снизилась на: {total_revenue_15 - total_revenue_14:,}")
print(f"Маржа выросла на: {total_margin_rub_15 - total_margin_rub_14:,}")
print(f"В процентном соотношении выручка снизилась на {diff_revenue:.2f}% темп прироста маржинальности составил {diff_margin:.2f}% за тот же период")




Неделя 14
Общая выручка: 8,411,011,220.0
Общая маржа (в деньгах): 1,230,219,074.0
Итоговая маржинальность: 14.63%

Неделя 15
Общая выручка: 8,156,720,245.0
Общая маржа (в деньгах): 1,261,407,562.0
Итоговая маржинальность: 15.46%

Diff
Выручка снизилась на: -254,290,975.0
Маржа выросла на: 31,188,488.0
В процентном соотношении выручка снизилась на -3.02% темп прироста маржинальности составил 2.54% за тот же период


На основе анализа можно сделать следующие выводы: снижение выручки некритично, это умеренное снижение, само оно не несет резко негативной картины, для полного понимания картины необходимо увеличть период анализа. Несмотря на то, что мы продали товаров на 254 млн. меньше, чистой прибыли мы заработали на 31 млн. больше. Это значит, что структура продаж сместилась в сторону более рентабельных товаров, либо мы подняли цены. 

In [6]:
pivot_df = pivot_df[pivot_df['Выручка_14'].notna() &
                    pivot_df['Выручка_15'].notna() &
                    pivot_df['Маржинальность_14'].notna() &
                    pivot_df['Маржинальность_15'].notna()]

In [7]:
pivot_df['темп_прирост_выручки_%'] = (pivot_df['Выручка_15'] - pivot_df['Выручка_14'])/pivot_df['Выручка_14'] * 100
pivot_df['темп_прирост_маржинальности_%'] = (pivot_df['Маржинальность_15'] - pivot_df['Маржинальность_14'])/pivot_df['Маржинальность_14'] * 100

In [8]:
pivot_df

,Бренд AX 1_,Выручка_14,Выручка_15,Маржинальность_14,Маржинальность_15,темп_прирост_выручки_%,темп_прирост_маржинальности_%
0,Ahava,44 410 651.58,73 153 235.14,0.12,0.08,64.72,-34.09
1,Anne Semonin,32 673 568.40,19 243 952.97,0.20,0.20,-41.10,-0.29
2,BABOR,309 282 334.73,290 841 977.18,0.15,0.15,-5.96,0.16
5,Biotherm,223 614 213.98,217 137 765.03,0.12,0.12,-2.90,-0.35
6,Botany,31 572 158.27,28 789 604.70,0.24,0.24,-8.81,1.15
7,CHANEL,236 703 280.29,217 319 320.17,0.09,0.08,-8.19,-11.94
9,COOLA,22 413 754.85,30 631 097.05,0.21,0.20,36.66,-1.10
10,Caudalie,175 729 536.95,220 140 721.46,0.19,0.17,25.27,-12.10
11,Cellcosmet,75 675 127.03,68 891 929.60,0.13,0.11,-8.96,-12.65
12,Christian Dior,148 429 286.65,129 024 994.36,0.09,0.11,-13.07,23.23


Разделим всю популяцию на несколько выборок, в зависимости от изменения выручки и маржинальности, и для каждой группы сделаем выводы

##### 1. изменения незначительны

In [9]:
revenue_makup = pivot_df[(pivot_df['темп_прирост_выручки_%'] <= 5) & 
                    (pivot_df['темп_прирост_выручки_%'] >= -5) &
                    (pivot_df['темп_прирост_маржинальности_%'] <= 3) &
                    (pivot_df['темп_прирост_маржинальности_%'] >= -3)

].copy()

revenue_makup

,Бренд AX 1_,Выручка_14,Выручка_15,Маржинальность_14,Маржинальность_15,темп_прирост_выручки_%,темп_прирост_маржинальности_%
5,Biotherm,223 614 213.98,217 137 765.03,0.12,0.12,-2.90,-0.35
20,Ecooking,60 800 265.72,61 331 195.52,0.31,0.30,0.87,-1.57
54,Zielinski&Rozen,188 905 732.52,191 514 856.42,0.21,0.21,1.38,1.38


##### 2. изменения выручки незначительны маржинальность выросла

In [10]:
revenue_makup_up = pivot_df[(pivot_df['темп_прирост_выручки_%'] <= 5) & 
                    (pivot_df['темп_прирост_выручки_%'] >= -5) &
                    (pivot_df['темп_прирост_маржинальности_%'] >= 3)

].copy()

revenue_makup_up

,Бренд AX 1_,Выручка_14,Выручка_15,Маржинальность_14,Маржинальность_15,темп_прирост_выручки_%,темп_прирост_маржинальности_%
38,Loccitane,102 221 899.25,101 412 743.00,0.13,0.14,-0.79,8.86


##### 3. изменения выручки незначительны маржинальность снизилась

In [11]:
revenue_makup_down = pivot_df[(pivot_df['темп_прирост_выручки_%'] <= 5) & 
                    (pivot_df['темп_прирост_выручки_%'] >= -5) &
                    (pivot_df['темп_прирост_маржинальности_%'] <= -3)

].copy()

revenue_makup_down

,Бренд AX 1_,Выручка_14,Выручка_15,Маржинальность_14,Маржинальность_15,темп_прирост_выручки_%,темп_прирост_маржинальности_%
15,Clinique,485 396 784.14,498 201 172.67,0.17,0.14,2.64,-17.44
21,Estee Lauder,328 740 954.37,325 930 625.42,0.11,0.10,-0.85,-3.38
23,FOREO,74 808 841.78,76 494 007.54,0.21,0.19,2.25,-12.29
49,Skyn Iceland,58 450 928.07,57 110 518.70,0.16,0.16,-2.29,-3.71


##### 4. выручка и маржинальность выросла

In [12]:
revenue_up_makup_up = pivot_df[(pivot_df['темп_прирост_выручки_%'] >= 5) & 
                    (pivot_df['темп_прирост_маржинальности_%'] >= 3)

].copy()

revenue_up_makup_up

,Бренд AX 1_,Выручка_14,Выручка_15,Маржинальность_14,Маржинальность_15,темп_прирост_выручки_%,темп_прирост_маржинальности_%
29,Guerlain,65 019 238.26,81 926 827.48,0.09,0.09,26.00,9.25
35,La Prairie,199 366 786.74,209 556 683.71,0.15,0.18,5.11,13.81
51,Swiss Line,19 473 518.39,24 324 787.03,0.18,0.19,24.91,5.70
52,Thalgo,63 404 689.15,77 089 936.99,0.14,0.17,21.58,15.31
53,Valmont,195 077 531.83,270 735 485.91,0.15,0.16,38.78,6.72


##### 5.выручка и маржинальность снизилась

In [13]:
revenue_down_makup_down = pivot_df[(pivot_df['темп_прирост_выручки_%'] <= -5) & 
                    (pivot_df['темп_прирост_маржинальности_%'] <= -3)

].copy()

revenue_down_makup_down

,Бренд AX 1_,Выручка_14,Выручка_15,Маржинальность_14,Маржинальность_15,темп_прирост_выручки_%,темп_прирост_маржинальности_%
7,CHANEL,236 703 280.29,217 319 320.17,0.09,0.08,-8.19,-11.94
11,Cellcosmet,75 675 127.03,68 891 929.60,0.13,0.11,-8.96,-12.65
16,Darphin,49 670 121.31,37 508 782.01,0.14,0.14,-24.48,-6.41
17,Dr Barbara Sturm,41 725 673.52,27 390 688.11,0.22,0.19,-34.36,-12.80
30,Helena Rubinstein,48 107 733.58,35 725 576.31,0.13,0.08,-25.74,-38.00
39,Lumene,86 436 738.70,70 834 556.72,0.14,0.13,-18.05,-4.68


##### 6. Маржинальность без сильных изменений, выручка снизилась

In [14]:
revenue_down_makup = pivot_df[(pivot_df['темп_прирост_выручки_%'] <= -5) & 
                    (pivot_df['темп_прирост_маржинальности_%'] <= 3) &
                    (pivot_df['темп_прирост_маржинальности_%'] >= -3)

].copy()

revenue_down_makup.sort_values(by='темп_прирост_выручки_%')

,Бренд AX 1_,Выручка_14,Выручка_15,Маржинальность_14,Маржинальность_15,темп_прирост_выручки_%,темп_прирост_маржинальности_%
1,Anne Semonin,32 673 568.40,19 243 952.97,0.20,0.20,-41.10,-0.29
22,FILORGA,60 160 904.53,43 946 338.03,0.21,0.21,-26.95,2.41
46,SENSAI,161 624 333.42,125 798 038.82,0.15,0.14,-22.17,-1.63
14,Cle De Peau,158 152 134.18,126 510 755.29,0.08,0.08,-20.01,0.39
40,MIXIT,29 108 357.12,23 972 143.55,0.18,0.17,-17.65,-1.35
44,Proraso,16 174 629.63,13 934 050.80,0.22,0.22,-13.85,-1.30
6,Botany,31 572 158.27,28 789 604.70,0.24,0.24,-8.81,1.15
47,Shiseido,1 353 685 520.77,1 267 702 457.57,0.17,0.17,-6.35,0.83
2,BABOR,309 282 334.73,290 841 977.18,0.15,0.15,-5.96,0.16


##### 7. выручка выросла маржинальность без сильных изменений

In [15]:
revenue_up_makup = pivot_df[(pivot_df['темп_прирост_выручки_%'] >= 5) & 
                    (pivot_df['темп_прирост_маржинальности_%'] <= 3) &
                    (pivot_df['темп_прирост_маржинальности_%'] >= -3)

].copy()

revenue_up_makup.sort_values(by='темп_прирост_выручки_%')

,Бренд AX 1_,Выручка_14,Выручка_15,Маржинальность_14,Маржинальность_15,темп_прирост_выручки_%,темп_прирост_маржинальности_%
28,Guam,116 255 405.20,124 585 676.50,0.15,0.15,7.17,1.38
45,Rexaline,79 162 873.03,86 082 707.41,0.13,0.13,8.74,-0.12
33,Kiehls,185 615 839.18,206 567 847.87,0.19,0.19,11.29,-0.45
19,ELEMIS,127 289 496.17,142 541 743.16,0.23,0.23,11.98,-0.44
26,Glam Glow,62 879 247.57,73 313 744.34,0.12,0.13,16.59,1.43
34,La Mer,228 013 867.60,274 024 565.47,0.17,0.17,20.18,0.34
31,James Read,44 271 548.63,54 699 468.03,0.23,0.23,23.55,0.20
9,COOLA,22 413 754.85,30 631 097.05,0.21,0.20,36.66,-1.10
41,OK Beauty,57 984 393.75,79 664 556.07,0.16,0.16,37.39,1.74
50,St.Barth,66 833 732.02,101 421 030.13,0.19,0.19,51.75,-1.41


##### 8. выручка выросла, маржиальность снизилась

In [16]:
revenue_up_makup_down = pivot_df[(pivot_df['темп_прирост_выручки_%'] >= 5) & 
                    (pivot_df['темп_прирост_маржинальности_%'] <= -3) 

].copy()

revenue_up_makup_down

,Бренд AX 1_,Выручка_14,Выручка_15,Маржинальность_14,Маржинальность_15,темп_прирост_выручки_%,темп_прирост_маржинальности_%
0,Ahava,44 410 651.58,73 153 235.14,0.12,0.08,64.72,-34.09
10,Caudalie,175 729 536.95,220 140 721.46,0.19,0.17,25.27,-12.10
32,Kenzo,30 500 823.75,39 593 275.37,0.09,0.05,29.81,-41.56
36,Lancaster,79 889 078.52,124 200 394.14,0.14,0.10,55.47,-31.14
42,Omorovicza,28 865 317.75,36 764 981.61,0.18,0.17,27.37,-5.42
43,Payot,265 832 651.84,368 126 150.03,0.22,0.21,38.48,-3.11


##### 9. выручка снизилась, маржиальность выросла

In [17]:
revenue_down_makup_up = pivot_df[(pivot_df['темп_прирост_выручки_%'] <= -5) & 
                    (pivot_df['темп_прирост_маржинальности_%'] >= 3) 

].copy()

revenue_down_makup_up

,Бренд AX 1_,Выручка_14,Выручка_15,Маржинальность_14,Маржинальность_15,темп_прирост_выручки_%,темп_прирост_маржинальности_%
12,Christian Dior,148 429 286.65,129 024 994.36,0.09,0.11,-13.07,23.23
13,Clarins,1 329 934 371.92,974 711 947.75,0.09,0.15,-26.71,59.72
25,Givenchy,17 931 399.81,15 352 372.07,0.11,0.11,-14.38,7.35
37,Lancome,241 346 125.35,204 288 646.00,0.10,0.13,-15.35,24.55
48,Sisley,301 387 817.96,206 654 314.05,0.16,0.17,-31.43,7.44


##### Аналитическая записка

Для написания аналитической записки мы разделили всю популяцию на 9 групп в зависимости от сценария изменения выручки и маржинальности.

| Сценарий | Выручка | Маржа | Бренды |
| :--- | :---: | :---: | :--- |
| **1** | 0 | 0 |Biotherm, Ecooking, Zielinski&Rozen |
| **2** | 0 | ↑ |Loccitane |
| **3** | 0 | ↓ |Clinique, FOREO, Skyn Iceland, Estee Lauder |
| **4** | ↑ | ↑ |Guerlain, La Prairie, Swiss Line, Thalgo, Valmont |
| **5** | ↓ | ↓ |CHANEL, Cellcosmet, Darphin, Dr Barbara Sturm, Helena Rubinstein, Lumene |
| **6** | ↓ | 0 |Botany, Shiseido, BABOR, Anne Semonin, FILORGA, SENSAI, Cle De Peau, MIXIT, Proraso |
| **7** | ↑ | 0 |Guam, Rexaline, Kiehls, ELEMIS, Glam Glow, La Mer, James Read, COOLA, OK Beauty, St.Barth |
| **8** | ↑ | ↓ |Ahava, Caudalie, Kenzo, Lancaster, Omorovicza, Payot |
| **9** | ↓ | ↑ |Christian Dior, Clarins, Givenchy, Lancome, Sisley |

В первый сценарий попали товары брендов Biotherm, Ecooking и Zielinski&Rozen незначительные колебания выручки и маржинальности нормальная ситуация для ритейлера. Однако необходимо отслеживать, чтобы снижение не было на протяжении длительного периода, что может сигнализировать об ухудшении позиции на рынке или других причинах снижения.

Во второй сценарий попал бренд Loccitane, сценарий при котором выручка практически не изменилась, а маржинальность выросла в нашем случае на 8.8%. Такое может произойти если цена поднялась, и снизилось кол-во продаж, однако по причине высокой цены выручка осталась на прежнем уровне. Для более точного анализа необходима информация о кол-ве проданного товара.

В третьем сценарии выручка практически не изменилась, однако маржинальность снизилась. В данную группу попали бренды Clinique и FOREO маржинальность снизилась на 17% и 12%, такое возможно если закупочная цена выросла, однако цена реализации осталось прежней. У брендов Skyn Iceland и Estee Lauder снижение маржинальности составило 3.7% и 3.4. Это очень близко к нашему порогу игнорирования изменения (3%) по данным брендам необходимо провести анализ за более длительный период чтобы сделать конкретные выводы.

Четвертый сценарий при котром оба показатели выросли возможен если мы подняли цены, однако спрос на него все равно вырос например сезонность или отсутвие у конкурентов, в данную группу попали товары брендов Guerlain, La Prairie, Swiss Line, Thalgo, Valmont.

В пятом сценарии снижение произошло и по выручке, и по маржинальности. В данную группу попали товары брендов CHANEL, Cellcosmet, Darphin, Dr Barbara Sturm, Helena Rubinstein, Lumene. Такой вариант мог произойти если, например, спрос на данные товары снизился, мы запустили скидочные акции, однако они не помогли поднть спрос и увеличить выручку.

В шестую группу попали сразу 9 брендов. Снижение выручки и стабильная маржинальность может свидетельствовать о снижении спроса и стабильных ценах либо отсутствии товаров в магазинах. Botany, Shiseido, BABOR по данным бренда снижение выручки составило максимум 8.8% это близко к нашему порогу игнорироания изменений выручки (5%) по данным брендам неободимо увеличить диапазон анализа, чтобы сделать выводы. Anne Semonin, FILORGA, SENSAI, Cle De Peau, MIXIT, Proraso снижение выручки существенное, необходимо больше информации, чтобы выявить причину снижения продаж.

В седьмом сценарии маржинальность практически не изменилась, однако выручка выросла. Бренды в данной группе можно разделить на две подгруппе. В первой выручка изменилась не значительно и необходим анализ за более продолжительный период, Guam (7.17%), Rexaline (8.74%). Во второй подгруппе бренды Kiehls, ELEMIS, Glam Glow, La Mer, James Read, COOLA, OK Beauty, St.Barth в данной группе рост выручки составил от 11.3% до 51.8%. Даннеы изменения возможны при росте спроса и неизменной наценки, возможно появились новинки с аналогичной наценкой реализация которых дала рост выручки. Либо поступление товаров в магазины где они кончились и не было реализации.

Восьмой сценарий при котором выручка растет, а маржинальность падает может произойти по причине проведения акций: снижение наценки увеличило спрос, однако маржинальность снизилась. В данную группу вошли следующие бренды Ahava, Caudalie, Kenzo, Lancaster, Omorovicza, Payot.

В дувятую группу, где выручка снижается, но растет маржинальность вошли бренды Christian Dior, Clarins, Givenchy, Lancome, Sisley. Такое могло произойти после повышения цен на данную продукцию: спрос снизился, однако маржинальность возросла.